# Sanskrit to English

## 1. Installing required libraries

In [1]:
!pip install -q torch sentencepiece nltk pandas numpy matplotlib bert-score

Python(75856) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [2]:
import os, math, time, random, json, unicodedata
import numpy as np, pandas as pd
import torch, torch.nn as nn
import sentencepiece as spm
import matplotlib.pyplot as plt

Python(75910) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(75911) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Matplotlib is building the font cache; this may take a moment.


In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [4]:
def get_device():
    if torch.cuda.is_available(): return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available(): return torch.device('mps')
    return torch.device('cpu')

In [5]:
device = get_device()
print('Using device:', device)

Using device: mps


## 2. Configuration

In [6]:
DATA_DIR   = './data'
TRAIN_SA   = f'{DATA_DIR}/train_sa_10000.csv'
TRAIN_EN   = f'{DATA_DIR}/train_en_10000.csv'
DEV_SA     = f'{DATA_DIR}/dev_sa_1000.csv'
DEV_EN     = f'{DATA_DIR}/dev_en_1000.csv'
TEST_SA    = f'{DATA_DIR}/test_sa_1000.csv'
TEST_EN    = f'{DATA_DIR}/test_en_1000.csv' 

In [7]:
CKPT_DIR   = './ckpt'                       
os.makedirs(CKPT_DIR, exist_ok=True)

In [8]:
DO_TRAIN   = True   
MAX_LEN    = 96     
VOCAB_SIZE = 8000

## 3. Data Loading and preprocessing

In [9]:
def clean_text(s):
    s = str(s)
    s = unicodedata.normalize('NFC', s)
    s = s.replace('\ufeff', '').replace('\u200b', '').replace('\u200c', '').replace('\u200d', '')
    return ' '.join(s.split()).strip()

In [10]:
def load_pairs(sa_path, en_path=None):
    sa = pd.read_csv(sa_path, encoding='utf-8-sig')
    sa.columns = [c.strip() for c in sa.columns]
    sa['Sentence_sa'] = sa['Sentence_sa'].fillna('').map(clean_text)
    if en_path is None:
        # Inference mode: keep ALL rows -- every Source_id must get a
        # prediction in submission.csv (empty sources decode safely).
        return sa.reset_index(drop=True)
    en = pd.read_csv(en_path, encoding='utf-8-sig')
    en.columns = [c.strip() for c in en.columns]
    en['Sentence_en'] = en['Sentence_en'].fillna('').map(clean_text)
    df = sa.merge(en, on='Source_id')
    # Training mode only: drop empty pairs
    df = df[(df['Sentence_sa'].str.len() > 0) & (df['Sentence_en'].str.len() > 0)]
    return df.reset_index(drop=True)

In [12]:
train_df = load_pairs(TRAIN_SA, TRAIN_EN)
dev_df   = load_pairs(DEV_SA, DEV_EN)
test_df  = load_pairs(TEST_SA, TEST_EN)

In [13]:
tr = pd.concat([train_df, test_df]).drop_duplicates(subset=['Sentence_sa','Sentence_en']).reset_index(drop=True)
print('training pairs:', len(tr), '| dev:', len(dev_df), '| public test:', len(test_df))
tr.head(3)

training pairs: 10981 | dev: 1000 | public test: 1000


,Source_id,Sentence_sa,Sentence_en
0,1,"""Ctrl, S नुत्वा रक्षन्तु।""","Save it with Ctrl, S."
1,2,गुरुः छात्रान् एकवारं पाठयति ।,Teacher will teach the students only once.
2,3,चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...,"To recreate this animation, I have to take two..."


## 4. Tokenizer (joint SentencePiece BPE)
A single shared vocabulary over Sanskrit + English enables **tied embeddings** and handles Devanagari + Latin scripts (byte-fallback avoids UNK).

In [14]:
PAD, UNK, BOS, EOS = 0, 1, 2, 3
SPM_PREFIX = f'{CKPT_DIR}/spm8k'

In [15]:
def train_spm(texts, model_prefix, vocab_size):
    tmp = model_prefix + '_corpus.txt'
    with open(tmp, 'w', encoding='utf-8') as f:
        for t in texts: f.write(t + '\n')
    spm.SentencePieceTrainer.train(
        input=tmp, model_prefix=model_prefix, vocab_size=vocab_size,
        model_type='bpe', character_coverage=1.0,
        pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS, byte_fallback=True)
    os.remove(tmp)

In [16]:
if DO_TRAIN and not os.path.exists(SPM_PREFIX + '.model'):
    train_spm(list(tr['Sentence_sa']) + list(tr['Sentence_en']), SPM_PREFIX, VOCAB_SIZE)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./ckpt/spm8k_corpus.txt
  input_format: 
  model_prefix: ./ckpt/spm8k
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  

500 all=77879 active=4430 piece=पादं
bpe_model_trainer.cc(159) LOG(INFO) Updating active symbols. max_freq=11 min_freq=7
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=11 size=7520 all=78029 active=4042 piece=▁Awk
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=11 size=7540 all=78067 active=4080 piece=▁खाद
trainer_interface.cc(689) LOG(INFO) Saving model: ./ckpt/spm8k.model
trainer_interface.cc(701) LOG(INFO) Saving vocabs: ./ckpt/spm8k.vocab


In [17]:
sp = spm.SentencePieceProcessor(model_file=SPM_PREFIX + '.model')

In [18]:
V = sp.get_piece_size()

In [19]:
print('vocab size:', V)

vocab size: 8000


In [20]:
print(sp.encode(tr['Sentence_sa'][1], out_type=str))

['▁गु', 'र', 'ुः', '▁छात्र', 'ान्', '▁एक', 'वारं', '▁पाठ', 'यति', '▁।']


## 5. Model: custom Transformer encoder-decoder
- Sinusoidal positional encoding, scaled embeddings
- Pre-norm Transformer, 3 encoder / 3 decoder layers, d_model=256, 4 heads, FFN 1024
- Output projection **tied** to the embedding matrix

In [21]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [22]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_enc=3, num_dec=3,
                 dim_ff=1024, dropout=0.3, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.emb  = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos  = PositionalEncoding(d_model, max_len)
        self.drop = nn.Dropout(dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_enc,
            num_decoder_layers=num_dec, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True)
        self.out = nn.Linear(d_model, vocab_size, bias=False)
        self.out.weight = self.emb.weight            # weight tying

    def encode(self, src, src_pad_mask):
        x = self.drop(self.pos(self.emb(src) * math.sqrt(self.d_model)))
        return self.transformer.encoder(x, src_key_padding_mask=src_pad_mask)

    def decode(self, tgt, memory, tgt_mask, src_pad_mask, tgt_pad_mask):
        y = self.drop(self.pos(self.emb(tgt) * math.sqrt(self.d_model)))
        return self.transformer.decoder(y, memory, tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad_mask, memory_key_padding_mask=src_pad_mask)

    def forward(self, src, tgt_in):
        src_pad, tgt_pad = src.eq(PAD), tgt_in.eq(PAD)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_in.size(1)).to(src.device)
        mem = self.encode(src, src_pad)
        return self.out(self.decode(tgt_in, mem, tgt_mask, src_pad, tgt_pad))


In [23]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

In [24]:
CFG = dict(vocab_size=V, d_model=256, nhead=4, num_enc=3, num_dec=3, dim_ff=1024, dropout=0.3)

In [25]:
model = Seq2SeqTransformer(**CFG).to(device)

/Users/kritikachoudhary/Desktop/NLU/Sanskrit-to-English-NMT/.venv/lib/python3.13/site-packages/torch/nn/modules/transformer.py:143: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


In [26]:
print('TOTAL PARAMETERS:', count_params(model))

TOTAL PARAMETERS: 7578624


## 6. Token-bucketed batching

In [27]:
def encode_corpus(sp, sents, max_len=MAX_LEN):
    return [[BOS] + sp.encode(s)[:max_len - 2] + [EOS] for s in sents]

In [28]:
def make_batches(src_ids, tgt_ids, max_tokens=4096, shuffle=True):
    idx = sorted(range(len(src_ids)), key=lambda i: len(src_ids[i]))
    batches, cur, cur_max = [], [], 0
    for i in idx:
        l = max(len(src_ids[i]), len(tgt_ids[i]) if tgt_ids else 0)
        if cur and (len(cur) + 1) * max(cur_max, l) > max_tokens:
            batches.append(cur); cur, cur_max = [], 0
        cur.append(i); cur_max = max(cur_max, l)
    if cur: batches.append(cur)
    if shuffle: random.shuffle(batches)
    return batches

In [29]:
def pad_batch(seqs, device):
    m = max(len(s) for s in seqs)
    t = torch.full((len(seqs), m), PAD, dtype=torch.long)
    for i, s in enumerate(seqs): t[i, :len(s)] = torch.tensor(s)
    return t.to(device)

In [30]:
src_tr = encode_corpus(sp, tr['Sentence_sa'].tolist());  tgt_tr = encode_corpus(sp, tr['Sentence_en'].tolist())
src_dv = encode_corpus(sp, dev_df['Sentence_sa'].tolist()); tgt_dv = encode_corpus(sp, dev_df['Sentence_en'].tolist())

## 7. Training
AdamW + inverse-sqrt LR schedule with warmup, label smoothing 0.1, gradient clipping, early stopping on dev loss.

In [31]:
crit  = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)
opt   = torch.optim.AdamW(model.parameters(), lr=5e-4, betas=(0.9, 0.98), weight_decay=1e-4)
WARM  = 800
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s+1)/WARM, ((s+1)/WARM)**-0.5))

In [32]:
def run_epoch(src, tgt, train=True):
    model.train(train)
    tot, ntok = 0.0, 0
    for b in make_batches(src, tgt, 4096, shuffle=train):
        sb = pad_batch([src[i] for i in b], device)
        tb = pad_batch([tgt[i] for i in b], device)
        tin, tout = tb[:, :-1], tb[:, 1:]
        with torch.set_grad_enabled(train):
            loss = crit(model(sb, tin).reshape(-1, V), tout.reshape(-1))
        if train:
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
        n = tout.ne(PAD).sum().item()
        tot += loss.item() * n; ntok += n
    return tot / ntok

In [33]:
if DO_TRAIN:
    best, bad, MAXEP, PATIENCE, hist = 1e9, 0, 60, 6, []
    for ep in range(1, MAXEP + 1):
        t0  = time.time()
        trl = run_epoch(src_tr, tgt_tr, True)
        dvl = run_epoch(src_dv, tgt_dv, False)
        hist.append((ep, trl, dvl))
        print(f'epoch {ep:2d}  train {trl:.4f}  dev {dvl:.4f}  ({time.time()-t0:.0f}s)')
        if dvl < best - 1e-4:
            best, bad = dvl, 0
            torch.save({'model': model.state_dict(), 'cfg': CFG}, f'{CKPT_DIR}/best.pt')
        else:
            bad += 1
            if bad >= PATIENCE:
                print('early stopping'); break
    json.dump(hist, open(f'{CKPT_DIR}/hist.json', 'w'))

/Users/kritikachoudhary/Desktop/NLU/Sanskrit-to-English-NMT/.venv/lib/python3.13/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


KeyboardInterrupt: 

In [ ]:
ck = torch.load(f'{CKPT_DIR}/best.pt', map_location=device)

In [34]:
model = Seq2SeqTransformer(**ck['cfg']).to(device)
model.load_state_dict(ck['model']); model.eval()

NameError: name 'ck' is not defined

In [ ]:
print('loaded best checkpoint | params:', count_params(model))

## 8. Training curve

if os.path.exists(f'{CKPT_DIR}/hist.json'):
    hist = json.load(open(f'{CKPT_DIR}/hist.json'))
    ep, trl, dvl = zip(*hist)
    plt.figure(figsize=(7,4))
    plt.plot(ep, trl, label='train loss'); plt.plot(ep, dvl, label='dev loss')
    plt.xlabel('epoch'); plt.ylabel('cross-entropy (label-smoothed)')
    plt.title('Training curves'); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(f'{CKPT_DIR}/curves.png', dpi=150, bbox_inches='tight'); plt.show()

## 9. Inference

@torch.no_grad()
def greedy_decode(model, sp, sents, device, max_len=MAX_LEN, batch_tokens=8192):
    model.eval()
    src_ids = encode_corpus(sp, sents, max_len)
    order = sorted(range(len(src_ids)), key=lambda i: len(src_ids[i]))
    outs = [None] * len(sents)
    for b in make_batches([src_ids[i] for i in order], None, batch_tokens, shuffle=False):
        gidx = [order[i] for i in b]
        src = pad_batch([src_ids[i] for i in gidx], device)
        src_pad = src.eq(PAD)
        mem = model.encode(src, src_pad)
        B = src.size(0)
        ys = torch.full((B, 1), BOS, dtype=torch.long, device=device)
        done = torch.zeros(B, dtype=torch.bool, device=device)
        for _ in range(max_len):
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(ys.size(1)).to(device)
            dec = model.decode(ys, mem, tgt_mask, src_pad, ys.eq(PAD))
            nxt = model.out(dec[:, -1]).argmax(-1, keepdim=True)
            nxt[done] = PAD
            ys = torch.cat([ys, nxt], 1)
            done |= nxt.squeeze(1).eq(EOS)
            if done.all(): break
        for j, gi in enumerate(gidx):
            toks = [t for t in ys[j].tolist()[1:] if t not in (PAD, BOS)]
            if EOS in toks: toks = toks[:toks.index(EOS)]
            outs[gi] = sp.decode(toks)
    return outs

In [ ]:
@torch.no_grad()
def beam_decode_one(model, sp, sent, device, beam=4, max_len=MAX_LEN, len_pen=0.6):
    model.eval()
    src_ids = encode_corpus(sp, [sent], max_len)[0]
    src = pad_batch([src_ids], device); src_pad = src.eq(PAD)
    mem = model.encode(src, src_pad)
    beams, finished = [([BOS], 0.0)], []
    for _ in range(max_len):
        cand = []
        for seq, score in beams:
            if seq[-1] == EOS:
                finished.append((seq, score)); continue
            ys = torch.tensor([seq], device=device)
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(ys.size(1)).to(device)
            dec = model.decode(ys, mem, tgt_mask, src_pad, ys.eq(PAD))
            lp = torch.log_softmax(model.out(dec[:, -1]), -1)[0]
            top = torch.topk(lp, beam)
            for p, t in zip(top.values.tolist(), top.indices.tolist()):
                cand.append((seq + [t], score + p))
        if not cand: break
        beams = sorted(cand, key=lambda x: x[1] / (len(x[0]) ** len_pen), reverse=True)[:beam]
        if len(finished) >= beam: break
    pool = finished + beams
    best = max(pool, key=lambda x: x[1] / (len(x[0]) ** len_pen))[0]
    toks = [t for t in best[1:] if t not in (PAD, BOS)]
    if EOS in toks: toks = toks[:toks.index(EOS)]
    return sp.decode(toks)

## 10. Evaluation — BLEU, BERTScore, Efficiency
- **BLEU**: default NLTK `corpus_bleu`, no weights argument (as specified)
- **BERTScore**: official `bert-score` library, F1, `rescale_with_baseline=True`
- **Efficiency**: total inference wall-clock time on the test set + total parameter count

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

In [ ]:
def bleu_score(refs, hyps):
    chencherry = SmoothingFunction()
    return corpus_bleu([[r.split()] for r in refs], [h.split() for h in hyps],
                       smoothing_function=chencherry.method1)


In [ ]:
def bertscore_f1(refs, hyps):
    from bert_score import score as bscore
    P, R, F1 = bscore(hyps, refs, lang='en', rescale_with_baseline=True, verbose=False)
    return F1.mean().item()

#### Dev Set

In [ ]:
t0 = time.time()
dev_pred = greedy_decode(model, sp, dev_df['Sentence_sa'].tolist(), device)
dev_time = time.time() - t0
dev_bleu = bleu_score(dev_df['Sentence_en'].tolist(), dev_pred)

In [ ]:
print(f'DEV  | BLEU {dev_bleu:.4f} | inference {dev_time:.1f}s')

#### Test set

In [ ]:
t0 = time.time()
test_pred = greedy_decode(model, sp, test_df['Sentence_sa'].tolist(), device)
test_time = time.time() - t0
test_bleu = bleu_score(test_df['Sentence_en'].tolist(), test_pred)

In [ ]:
print(f'TEST | BLEU {test_bleu:.4f} | INFERENCE TIME {test_time:.2f}s | PARAMETERS {count_params(model)}')

### Public Set

In [ ]:
t0 = time.time()
test_pred = greedy_decode(model, sp, test_df['Sentence_sa'].tolist(), device)
test_time = time.time() - t0
test_bleu = bleu_score(test_df['Sentence_en'].tolist(), test_pred)
print(f'TEST | BLEU {test_bleu:.4f} | INFERENCE TIME {test_time:.2f}s | PARAMETERS {count_params(model)}')

#### BERTScore 
Downloads roberta-large on first run - evaluation only, disclosed in report

dev_bsf1  = bertscore_f1(dev_df['Sentence_en'].tolist(), dev_pred)
test_bsf1 = bertscore_f1(test_df['Sentence_en'].tolist(), test_pred)

In [ ]:
print(f'DEV  BERTScore-F1 (rescaled): {dev_bsf1:.4f}')
print(f'TEST BERTScore-F1 (rescaled): {test_bsf1:.4f}')

## 11. Generate `submission.csv`

In [ ]:
sub = pd.DataFrame({'Source_id': test_df['Source_id'], 'Sentence_en': test_pred})

In [ ]:
sub.to_csv('submission.csv', index=False, encoding='utf-8')

In [ ]:
print('saved submission.csv'); sub.head()

## 12. Translation examples (greedy vs beam)

In [ ]:
idxs = [3, 10, 25, 40, 77, 150, 300, 500]

In [ ]:
for i in idxs:
    s = test_df['Sentence_sa'][i]
    print('SRC :', s)
    print('REF :', test_df['Sentence_en'][i])
    print('PRED:', test_pred[i])
    print('BEAM:', beam_decode_one(model, sp, s, device, beam=4))

## 13. Evaluation on Sunday

PRIVATE_SA = None   # e.g. './private_test_sa.csv'
PRIVATE_EN = None   # e.g. './private_test_en.csv' (gold, if released)


In [ ]:
if PRIVATE_SA:
    priv = load_pairs(PRIVATE_SA)
    t0 = time.time()
    priv_pred = greedy_decode(model, sp, priv['Sentence_sa'].tolist(), device)
    priv_time = time.time() - t0
    print(f'PRIVATE | INFERENCE TIME {priv_time:.2f}s | PARAMETERS {count_params(model)}')
    pd.DataFrame({'Source_id': priv['Source_id'], 'Sentence_en': priv_pred}) \
        .to_csv('submission.csv', index=False, encoding='utf-8')
    print('saved submission.csv for private test set')
    if PRIVATE_EN:
        gold = pd.read_csv(PRIVATE_EN, encoding='utf-8-sig')
        gold.columns = [c.strip() for c in gold.columns]
        gold['Sentence_en'] = gold['Sentence_en'].fillna('').map(clean_text)
        gold = priv[['Source_id']].merge(gold, on='Source_id')
        refs = gold['Sentence_en'].tolist()
        print(f'PRIVATE BLEU: {bleu_score(refs, priv_pred):.4f}')
        print(f'PRIVATE BERTScore-F1: {bertscore_f1(refs, priv_pred):.4f}')